In [1]:
# ============================================================
# Appendix 9.3 Reproducibility Builder
# Key-observation robustness support (Tables 10–15)
#
# Reads from:
#   C:\Android Mobile App\ICST2026_Ext\MainDataset.csv
#
# Writes to:
#   C:\Android Mobile App\ICST2026_Ext\3.0-Observations\robustness_check
#
# Outputs:
#   - table10_step3_robustness_support.csv/.md
#   - table11_tier1_support.csv/.md
#   - table12_tier2_support.csv/.md
#   - table13_overall_robustness_summary.csv/.md
#   - table14_tier1_tests.csv/.md
#   - table15_tier2_tests.csv/.md
#   - appendix_9_3_bundle.xlsx
#   - appendix_9_3_run_notes.txt
#
# Notes
# -----
# - This script is designed for the current MainDataset schema.
# - It uses MainDataset only. The current dataset already contains:
#     * Base_timing_regime
#     * Layer2_available_in_base
#     * study_signature_hash
#     * coarsened_family__<signature>
#   so the stage-3 zip is not required here.
# - It prints all generated tables to the console and saves them.
# ============================================================

from __future__ import annotations

from pathlib import Path
import math
import warnings
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu, chi2_contingency

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
IN_MAIN = BASE_DIR / "MainDataset.csv"
OUT_DIR = BASE_DIR / r"3.0-Observations\robustness_check"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
STYLE_ORDER = ["Community", "GMD", "Third-Party", "Custom"]
TIMING_ANCHOR = "88f32b360855c277"
SECONDARY_ANCHOR = "0bc0e933a2435166"

TABLE10_NAME = "table10_step3_robustness_support"
TABLE11_NAME = "table11_tier1_support"
TABLE12_NAME = "table12_tier2_support"
TABLE13_NAME = "table13_overall_robustness_summary"
TABLE14_NAME = "table14_tier1_tests"
TABLE15_NAME = "table15_tier2_tests"

MEASURE_COLS = {
    "run_duration": "study_run_duration_seconds",
    "l1_time_to_instr": "study_layer1_time_to_instrumentation_envelope_seconds",
    "l1_instr_env": "study_layer1_instrumentation_job_envelope_seconds",
    "l1_post_tail": "study_layer1_post_instrumentation_tail_seconds",
    "l2_pre": "study_pre_invocation_selected_stage3_seconds",
    "l2_exec": "study_invocation_execution_window_selected_stage3_seconds",
    "l2_post": "study_post_invocation_selected_stage3_seconds",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def norm_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
        "y": True,
        "n": False,
        "base": True,
        "non-base": False,
    }
    return s.map(mapping).fillna(False).astype(bool)

def to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def canon_style(x) -> str | None:
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    mapping = {
        "community": "Community",
        "gmd": "GMD",
        "third-party": "Third-Party",
        "third party": "Third-Party",
        "custom": "Custom",
    }
    return mapping.get(s.lower(), s)

def cliff_delta(x: pd.Series, y: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna().to_numpy()
    y = pd.to_numeric(y, errors="coerce").dropna().to_numpy()
    if len(x) == 0 or len(y) == 0:
        return np.nan
    # rank-based shortcut from U:
    u = mannwhitneyu(x, y, alternative="two-sided").statistic
    return float((2 * u) / (len(x) * len(y)) - 1)

def cramer_v_from_table(ct: pd.DataFrame) -> float:
    if ct.empty:
        return np.nan
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.to_numpy().sum()
    if n == 0:
        return np.nan
    r, k = ct.shape
    return float(np.sqrt(chi2 / (n * max(1, min(r - 1, k - 1)))))

def fmt_p(p: float) -> str:
    if pd.isna(p):
        return "--"
    return f"{p:.2e}".replace("e-0", "e-").replace("e+0", "e+")

def fmt_delta(d: float) -> str:
    if pd.isna(d):
        return "--"
    return f"{d:.3f}"

def fmt_v(v: float) -> str:
    if pd.isna(v):
        return "--"
    return f"{v:.3f}"

def fmt_num(x, digits=3):
    if pd.isna(x):
        return "--"
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    return f"{x:.{digits}f}"

def median_style_order(df_part: pd.DataFrame, measure_col: str, min_n: int = 10) -> list[str]:
    rows = []
    for style in STYLE_ORDER:
        s = to_num(df_part.loc[df_part["style"] == style, measure_col]).dropna()
        if len(s) >= min_n:
            rows.append((style, float(s.median())))
    rows = sorted(rows, key=lambda t: (t[1], STYLE_ORDER.index(t[0])))
    return [x[0] for x in rows]

def compare_orderings(reference_order: list[str], other_order: list[str]) -> str:
    if len(other_order) < 2:
        return "not testable"
    if other_order == reference_order:
        return "preserved exactly"
    if len(reference_order) >= 1 and len(other_order) >= 1 and reference_order[0] == other_order[0]:
        return "partially preserved"
    if len(reference_order) >= 2 and len(other_order) >= 2 and set(reference_order[:2]) == set(other_order[:2]):
        return "partially preserved"
    return "changed"

def ensure_columns(df: pd.DataFrame, cols: list[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def save_table(df: pd.DataFrame, stem: str) -> None:
    csv_path = OUT_DIR / f"{stem}.csv"
    md_path = OUT_DIR / f"{stem}.md"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    try:
        md_text = df.to_markdown(index=False)
    except Exception:
        md_text = df.to_string(index=False)
    md_path.write_text(md_text, encoding="utf-8")
    print("\n" + "=" * 100)
    print(stem.upper())
    print("=" * 100)
    print(df.to_string(index=False))

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
if not IN_MAIN.exists():
    raise FileNotFoundError(f"MainDataset.csv not found: {IN_MAIN}")

df = pd.read_csv(IN_MAIN, low_memory=False)

# Normalize key fields
df["style"] = df["style"].map(canon_style)
df["run_attempt"] = to_num(df["run_attempt"])
df["controller_usable_verdict"] = norm_bool(df["controller_usable_verdict"])
df["controller_attempt_eq_1"] = norm_bool(df["controller_attempt_eq_1"])
df["controller_instru_job_count_gt0"] = norm_bool(df["controller_instru_job_count_gt0"])
df["controller_style_in_scope"] = norm_bool(df["controller_style_in_scope"])
df["Base_timing_regime"] = norm_bool(df["Base_timing_regime"])
df["Layer2_available_in_base"] = norm_bool(df["Layer2_available_in_base"])
df["Step_telemetry"] = norm_bool(df["Step_telemetry"])

df["run_conclusion_norm"] = df["run_conclusion"].astype(str).str.strip().str.lower()
df["event_norm"] = df["event"].astype(str).str.strip()

for c in MEASURE_COLS.values():
    if c in df.columns:
        df[c] = to_num(df[c])

ensure_columns(
    df,
    [
        "style",
        "study_signature_hash",
        "study_runner_os_bucket",
        "study_job_count_exec_bucket",
        "study_step_count_exec_bucket",
        "Base_timing_regime",
        "Layer2_available_in_base",
        "controller_attempt_eq_1",
        "controller_usable_verdict",
        "controller_instru_job_count_gt0",
        "controller_style_in_scope",
        "run_attempt",
        "run_conclusion_norm",
        "event_norm",
    ] + list(MEASURE_COLS.values())
)

# ------------------------------------------------------------
# Regime masks
# ------------------------------------------------------------
mask_all = (
    df["controller_style_in_scope"]
    & df["controller_instru_job_count_gt0"]
    & df["style"].isin(STYLE_ORDER)
)

mask_first_attempt = mask_all & df["controller_attempt_eq_1"]
mask_base = df["Base_timing_regime"]
mask_rerun_usable = mask_all & df["controller_usable_verdict"] & (df["run_attempt"] > 1)
mask_l2_in_base = df["Layer2_available_in_base"]

# ------------------------------------------------------------
# Utility: measure subsets
# ------------------------------------------------------------
def subset_for_measure(mask: pd.Series, measure_key: str) -> pd.DataFrame:
    out = df.loc[mask].copy()
    col = MEASURE_COLS[measure_key]
    if measure_key.startswith("l2_"):
        out = out.loc[out["Layer2_available_in_base"]].copy()
    out = out.loc[out["style"].isin(STYLE_ORDER)].copy()
    out = out.loc[out[col].notna()].copy()
    return out

# ------------------------------------------------------------
# Table 10 — Step 3 robustness support
# ------------------------------------------------------------
timing_measure_keys = [
    "run_duration",
    "l1_time_to_instr",
    "l1_instr_env",
    "l1_post_tail",
    "l2_pre",
    "l2_exec",
    "l2_post",
]

base_orders = {}
for mk in timing_measure_keys:
    ref_df = subset_for_measure(mask_base, mk)
    base_orders[mk] = median_style_order(ref_df, MEASURE_COLS[mk], min_n=10)

def ordering_counts(target_mask: pd.Series):
    exact = partial = changed = not_testable = 0
    for mk in timing_measure_keys:
        ref = base_orders[mk]
        tgt_df = subset_for_measure(target_mask, mk)
        tgt = median_style_order(tgt_df, MEASURE_COLS[mk], min_n=10)
        status = compare_orderings(ref, tgt)
        if status == "preserved exactly":
            exact += 1
        elif status == "partially preserved":
            partial += 1
        elif status == "changed":
            changed += 1
        else:
            not_testable += 1
    return exact, partial, changed, not_testable

all_counts = ordering_counts(mask_all)
first_counts = ordering_counts(mask_first_attempt)
rerun_counts = ordering_counts(mask_rerun_usable)

# Candidate signatures under Base
base_sig_pool = df.loc[
    mask_base
    & df["study_signature_hash"].notna()
    & df["study_signature_hash"].astype(str).str.strip().ne("")
].copy()

candidate_sig_count = base_sig_pool["study_signature_hash"].nunique()

# Eligible signatures:
# Use explicit eligible_signature if present; otherwise reconstruct:
# total >= 80 and at least two styles with >= 15 records each
if "eligible_signature" in df.columns:
    elig = norm_bool(df["eligible_signature"])
    eligible_sigs = (
        df.loc[mask_base & elig & df["study_signature_hash"].notna(), "study_signature_hash"]
        .drop_duplicates()
        .tolist()
    )
else:
    sig_style_counts = (
        base_sig_pool.groupby(["study_signature_hash", "style"])
        .size()
        .rename("n")
        .reset_index()
    )
    eligible_sigs = []
    for sig, part in sig_style_counts.groupby("study_signature_hash"):
        total = int(part["n"].sum())
        usable_styles = int((part["n"] >= 15).sum())
        if total >= 80 and usable_styles >= 2:
            eligible_sigs.append(sig)

eligible_sigs = sorted(set(eligible_sigs))
n_eligible = len(eligible_sigs)

# Exact/family follow-up counts vs Base
exact_exact = exact_partial = exact_changed = 0
family_exact = family_partial = family_changed = 0

# compute only on eligible signatures; compare against Base order for each measure
for sig in eligible_sigs:
    exact_mask = mask_base & df["study_signature_hash"].eq(sig)

    family_col = f"coarsened_family__{sig}"
    if family_col in df.columns:
        family_mask = mask_base & norm_bool(df[family_col])
    else:
        # fallback to exact only if family flag missing
        family_mask = exact_mask.copy()

    for mk in timing_measure_keys:
        ref = base_orders[mk]

        exact_df = subset_for_measure(exact_mask, mk)
        exact_order = median_style_order(exact_df, MEASURE_COLS[mk], min_n=15)
        exact_status = compare_orderings(ref, exact_order)
        if exact_status == "preserved exactly":
            exact_exact += 1
        elif exact_status == "partially preserved":
            exact_partial += 1
        elif exact_status == "changed":
            exact_changed += 1

        fam_df = subset_for_measure(family_mask, mk)
        fam_order = median_style_order(fam_df, MEASURE_COLS[mk], min_n=15)
        fam_status = compare_orderings(ref, fam_order)
        if fam_status == "preserved exactly":
            family_exact += 1
        elif fam_status == "partially preserved":
            family_partial += 1
        elif fam_status == "changed":
            family_changed += 1

# candidate-pool sentence
sig_style_support = (
    base_sig_pool.groupby(["study_signature_hash", "style"])
    .size()
    .rename("n")
    .reset_index()
)
if eligible_sigs:
    usable_style_counts = []
    for sig in eligible_sigs:
        part = sig_style_support[sig_style_support["study_signature_hash"] == sig]
        usable_style_counts.append(int((part["n"] >= 15).sum()))
    usable_style_summary = sorted(set(usable_style_counts))
    if len(usable_style_summary) == 1:
        usable_style_phrase = f"each provides only {usable_style_summary[0]} usable styles"
    else:
        usable_style_phrase = "eligible signatures provide varying usable-style support"
else:
    usable_style_phrase = "no eligible signatures were identified"

table10 = pd.DataFrame([
    {
        "Check": "Controller regime",
        "Target": "all_run_per_style vs. Base",
        "Quantitative support": f"Base ordering preserved exactly for {all_counts[0]}/7 timing measures; {all_counts[1]}/7 partially preserved",
        "Result": "Largely preserved" if all_counts[0] >= 6 else "Mixed",
    },
    {
        "Check": "Controller regime",
        "Target": "first-attempt view vs. Base",
        "Quantitative support": f"Base ordering preserved exactly for {first_counts[0]}/7 timing measures; {first_counts[1]}/7 partially preserved",
        "Result": "Largely preserved" if first_counts[0] >= 6 else "Mixed",
    },
    {
        "Check": "Controller regime",
        "Target": "rerun_usable_verdict vs. Base",
        "Quantitative support": f"Base ordering preserved for {rerun_counts[0]}/7 measures; {rerun_counts[1]}/7 partially preserved; {rerun_counts[2]}/7 changed; {rerun_counts[3]}/7 not testable",
        "Result": "Distinct adjacent regime",
    },
    {
        "Check": "Workflow shape",
        "Target": "Exact-signature candidate pool",
        "Quantitative support": f"{candidate_sig_count} candidate signatures identified; only {n_eligible} eligible for deeper follow-up; {usable_style_phrase}, not a balanced four-style setting",
        "Result": "Partially sufficient" if n_eligible > 0 else "Insufficient",
    },
    {
        "Check": "Workflow shape",
        "Target": "Selected exact-signature follow-up",
        "Quantitative support": f"Across the {n_eligible} eligible signatures ({7 * n_eligible} within-signature timing checks), {exact_exact}/{7 * n_eligible} reproduce the full Base ordering exactly; {exact_partial}/{7 * n_eligible} are partially preserved; and {exact_changed}/{7 * n_eligible} differ from the Base pattern",
        "Result": "Informative, not standalone",
    },
    {
        "Check": "Workflow shape",
        "Target": "Selected paired-family follow-up",
        "Quantitative support": f"Across the {n_eligible} paired Tier 2 families ({7 * n_eligible} within-family timing checks), {family_exact}/{7 * n_eligible} reproduce the full Base ordering exactly; {family_partial}/{7 * n_eligible} are partially preserved; and {family_changed}/{7 * n_eligible} differ from the Base pattern",
        "Result": "Informative, not standalone",
    },
])

# ------------------------------------------------------------
# Observation specs
# ------------------------------------------------------------
OBS_META = {
    "1.1": {
        "title": "Community remains the fastest completion-oriented style, with clearer support against Third-Party than against GMD.",
        "tier1_direction": "Community remains lower than GMD and Third-Party on run duration and L2 execution window.",
        "tier2_direction": "Community remains faster than Third-Party and remains lower than GMD on the main completion-oriented measures.",
    },
    "1.2": {
        "title": "GMD remains the clearest fast-entry style relative to Third-Party, but the Community--GMD entry contrast stays weak.",
        "tier1_direction": "GMD remains earliest on L2 pre-invocation relative to Third-Party and remains not-fastest on completion; the Community-vs-GMD entry contrast is weaker.",
        "tier2_direction": "The fast-entry interpretation remains clear relative to Third-Party, but Community and GMD become close on the entry-side metric.",
    },
    "1.3": {
        "title": "Third-Party remains the slowest sustained-execution style across the key timing measures.",
        "tier1_direction": "Third-Party remains slowest on run duration, L2 pre-invocation, and L2 execution window.",
        "tier2_direction": "The slow sustained-execution interpretation remains visible on run duration, L2 pre-invocation, and L2 execution window.",
    },
    "2.1": {
        "title": "GMD remains the tightest and most predictable style on the main completion-oriented measures.",
        "tier1_direction": "GMD remains the tightest style on the main completion-oriented predictability measures.",
        "tier2_direction": "GMD remains the tightest style on the main completion-oriented predictability measures.",
    },
    "2.2": {
        "title": "Community remains fast in typical terms but substantially less predictable than GMD.",
        "tier1_direction": "Community remains fast in typical terms, but less stable than GMD on the main completion-oriented measures.",
        "tier2_direction": "Community remains fast in typical terms but predictability-poor relative to GMD.",
    },
    "3.1": {
        "title": "GMD remains the clearest execution-centric overhead profile, with a dominant execution share and minimal residual tail.",
        "tier1_direction": "GMD remains the clearest execution-centric style, with high execution share and very small residual tail.",
        "tier2_direction": "GMD remains the clearest execution-centric style in the broader family.",
    },
    "3.2": {
        "title": "Third-Party remains a heavy-entry plus heavy-execution style rather than a completion-tail-dominated one.",
        "tier1_direction": "Third-Party remains a heavy-entry plus heavy-execution style, not a completion-tail-dominated one.",
        "tier2_direction": "Third-Party remains characterized by heavy entry and heavy execution.",
    },
    "4.2": {
        "title": "The strong success-rate separation remains visible, but support is structurally split across anchors/families rather than unified in one balanced four-style setting.",
        "tier1_direction": "The strong success-rate separation remains visible, but no single exact signature supports a balanced four-style exact check.",
        "tier2_direction": "Success-rate differences remain visible, but support remains distributed across the two broadened families rather than one unified Tier 2 setting.",
    },
    "4.3": {
        "title": "The deployment-context separation remains clearly visible across the robustness follow-up settings.",
        "tier1_direction": "The trigger-context separation remains visible: Community aligns with push, GMD/Third-Party with schedule, and Custom with pull_request.",
        "tier2_direction": "The deployment-context pattern remains clearly visible in the paired broadened families.",
    },
}

# ------------------------------------------------------------
# Subset builders for observations
# ------------------------------------------------------------
def exact_subset(signature: str, rq4: bool = False, require_usable: bool = False) -> pd.DataFrame:
    mask = mask_first_attempt if rq4 else mask_base
    out = df.loc[mask & df["study_signature_hash"].eq(signature)].copy()
    if require_usable:
        out = out.loc[out["controller_usable_verdict"]].copy()
    return out

def family_subset(signature: str, rq4: bool = False, require_usable: bool = False) -> pd.DataFrame:
    mask = mask_first_attempt if rq4 else mask_base
    family_col = f"coarsened_family__{signature}"
    if family_col not in df.columns:
        raise ValueError(f"Missing family flag column: {family_col}")
    out = df.loc[mask & norm_bool(df[family_col])].copy()
    if require_usable:
        out = out.loc[out["controller_usable_verdict"]].copy()
    return out

# ------------------------------------------------------------
# Derived measures used in tests
# ------------------------------------------------------------
def add_share_columns(part: pd.DataFrame) -> pd.DataFrame:
    out = part.copy()
    rd = to_num(out[MEASURE_COLS["run_duration"]])
    for key in ["l2_pre", "l2_exec", "l2_post"]:
        col = MEASURE_COLS[key]
        out[f"{key}_share"] = to_num(out[col]) / rd
    return out

def add_norm_abs_dev(part: pd.DataFrame, measure_col: str, new_col: str) -> pd.DataFrame:
    out = part.copy()
    med_by_style = (
        out.groupby("style")[measure_col]
        .median()
        .rename("_med_style")
        .reset_index()
    )
    out = out.merge(med_by_style, on="style", how="left")
    out[new_col] = (to_num(out[measure_col]) - to_num(out["_med_style"])).abs() / to_num(out["_med_style"])
    out = out.drop(columns=["_med_style"])
    return out

# ------------------------------------------------------------
# Statistical test runners
# ------------------------------------------------------------
def mwu_row(
    obs: str,
    metric_label: str,
    contrast_label: str,
    part: pd.DataFrame,
    style_1: str,
    style_2: str,
    value_col: str,
    interpretation: str,
) -> dict:
    x = to_num(part.loc[part["style"] == style_1, value_col]).dropna()
    y = to_num(part.loc[part["style"] == style_2, value_col]).dropna()

    if len(x) == 0 or len(y) == 0:
        p = np.nan
        d = np.nan
        med1 = np.nan
        med2 = np.nan
    else:
        p = float(mannwhitneyu(x, y, alternative="two-sided").pvalue)
        d = cliff_delta(x, y)
        med1 = float(x.median())
        med2 = float(y.median())

    return {
        "Obs.": obs,
        "Metric": metric_label,
        "Contrast": contrast_label,
        "n1 / n2": f"{len(x)} / {len(y)}",
        "Median1 / Median2": f"{fmt_num(med1, 3)} / {fmt_num(med2, 3)}",
        "p / effect": f"p = {fmt_p(p)}, δ = {fmt_delta(d)}",
        "Interpretation": interpretation,
        "_p": p,
        "_effect": d,
        "_n1": len(x),
        "_n2": len(y),
    }

def chi_row(
    obs: str,
    metric_label: str,
    part_by_signature: list[tuple[str, pd.DataFrame]],
    col_label: str,
    interpretation: str,
) -> dict:
    ns = []
    chunks = []
    for sig, part in part_by_signature:
        ct = pd.crosstab(part["style"], part[col_label])
        ns.append(str(int(ct.to_numpy().sum())))
        if ct.empty or ct.shape[0] < 2 or ct.shape[1] < 2:
            chunks.append(f"{sig[:5]}: p = --, V = --")
        else:
            p = float(chi2_contingency(ct)[1])
            v = cramer_v_from_table(ct)
            chunks.append(f"{sig[:5]}: p = {fmt_p(p)}, V = {fmt_v(v)}")

    return {
        "Obs.": obs,
        "Metric": metric_label,
        "Contrast": f"Style × {col_label}",
        "n1 / n2": "; ".join(ns),
        "Median1 / Median2": "--",
        "p / effect": "; ".join(chunks),
        "Interpretation": interpretation,
    }

# ------------------------------------------------------------
# Build Tier 1 tests (Table 14)
# ------------------------------------------------------------
tier1_tests = []

# timing anchor
t1 = add_share_columns(exact_subset(TIMING_ANCHOR, rq4=False))
t1 = add_norm_abs_dev(t1, MEASURE_COLS["run_duration"], "nad_run_duration")
t1 = add_norm_abs_dev(t1, MEASURE_COLS["l2_exec"], "nad_l2_exec")

tier1_tests.append(
    mwu_row(
        "1.1",
        "Run duration",
        "Community vs. GMD",
        t1,
        "Community",
        "GMD",
        MEASURE_COLS["run_duration"],
        "Direction is preserved, but the completion-side separation between Community and GMD is weak in the exact-signature setting.",
    )
)
tier1_tests.append(
    mwu_row(
        "1.1",
        "L2 exec. window",
        "Community vs. Third-Party",
        t1,
        "Community",
        "Third-Party",
        MEASURE_COLS["l2_exec"],
        "Strong completion-side support remains visible relative to Third-Party.",
    )
)
tier1_tests.append(
    mwu_row(
        "1.2",
        "L2 pre-invocation",
        "GMD vs. Third-Party",
        t1,
        "GMD",
        "Third-Party",
        MEASURE_COLS["l2_pre"],
        "Fast-entry support is extremely strong relative to Third-Party.",
    )
)
tier1_tests.append(
    mwu_row(
        "1.2",
        "L2 pre-invocation",
        "Community vs. GMD",
        t1,
        "Community",
        "GMD",
        MEASURE_COLS["l2_pre"],
        "The Community-vs.-GMD entry contrast is weak in the exact-signature setting.",
    )
)
tier1_tests.append(
    mwu_row(
        "1.3",
        "Run duration",
        "Third-Party vs. Community",
        t1,
        "Third-Party",
        "Community",
        MEASURE_COLS["run_duration"],
        "Strong slow-path support for Third-Party is preserved in Tier 1.",
    )
)
tier1_tests.append(
    mwu_row(
        "1.3",
        "L2 exec. window",
        "Third-Party vs. GMD",
        t1,
        "Third-Party",
        "GMD",
        MEASURE_COLS["l2_exec"],
        "Sustained-execution disadvantage remains clearly visible against GMD.",
    )
)
tier1_tests.append(
    mwu_row(
        "2.1",
        "Norm. abs. deviation (run duration)",
        "GMD vs. Community",
        t1,
        "GMD",
        "Community",
        "nad_run_duration",
        "Strong Tier 1 predictability support: GMD remains much tighter than Community.",
    )
)
tier1_tests.append(
    mwu_row(
        "2.1",
        "Norm. abs. deviation (L2 exec. window)",
        "GMD vs. Third-Party",
        t1,
        "GMD",
        "Third-Party",
        "nad_l2_exec",
        "Predictability support remains clearly visible against Third-Party.",
    )
)
tier1_tests.append(
    mwu_row(
        "2.2",
        "Norm. abs. deviation (run duration)",
        "Community vs. GMD",
        t1,
        "Community",
        "GMD",
        "nad_run_duration",
        "The fast-but-variable Community profile remains strongly preserved in Tier 1.",
    )
)
tier1_tests.append(
    mwu_row(
        "3.1",
        "L2 execution share",
        "GMD vs. Community",
        t1,
        "GMD",
        "Community",
        "l2_exec_share",
        "Execution-centric separation remains very strong in the exact-signature setting.",
    )
)
tier1_tests.append(
    mwu_row(
        "3.1",
        "L2 post-invocation share",
        "Community vs. GMD",
        t1,
        "Community",
        "GMD",
        "l2_post_share",
        "Residual-tail contrast remains very strong against GMD.",
    )
)
tier1_tests.append(
    mwu_row(
        "3.2",
        "L2 pre-invocation share",
        "Third-Party vs. GMD",
        t1,
        "Third-Party",
        "GMD",
        "l2_pre_share",
        "Heavy-entry support for Third-Party remains very strong.",
    )
)
tier1_tests.append(
    mwu_row(
        "3.2",
        "L2 execution share",
        "Third-Party vs. Community",
        t1,
        "Third-Party",
        "Community",
        "l2_exec_share",
        "Heavy-execution placement remains preserved in the exact-signature setting.",
    )
)

# RQ4 split-anchor exact tests
obs42_t1_parts = [
    (TIMING_ANCHOR, exact_subset(TIMING_ANCHOR, rq4=True, require_usable=True)),
    (SECONDARY_ANCHOR, exact_subset(SECONDARY_ANCHOR, rq4=True, require_usable=True)),
]
for _, part in obs42_t1_parts:
    part["success_failure"] = np.where(part["run_conclusion_norm"] == "success", "success", "failure")
tier1_tests.append(
    chi_row(
        "4.2",
        "Success among usable verdicts",
        obs42_t1_parts,
        "success_failure",
        "Tier 1 exact-signature support for the success-rate separation is strong in the main anchor and weaker in the secondary anchor.",
    )
)

obs43_t1_parts = [
    (TIMING_ANCHOR, exact_subset(TIMING_ANCHOR, rq4=True, require_usable=False)),
    (SECONDARY_ANCHOR, exact_subset(SECONDARY_ANCHOR, rq4=True, require_usable=False)),
]
tier1_tests.append(
    chi_row(
        "4.3",
        "Trigger event distribution",
        obs43_t1_parts,
        "event_norm",
        "The deployment-context separation remains strongly supported across both eligible exact-signature settings.",
    )
)

table14 = pd.DataFrame(tier1_tests)

# ------------------------------------------------------------
# Build Tier 2 tests (Table 15)
# ------------------------------------------------------------
tier2_tests = []

t2 = add_share_columns(family_subset(TIMING_ANCHOR, rq4=False))
t2 = add_norm_abs_dev(t2, MEASURE_COLS["run_duration"], "nad_run_duration")
t2 = add_norm_abs_dev(t2, MEASURE_COLS["l2_exec"], "nad_l2_exec")

tier2_tests.append(
    mwu_row(
        "1.1",
        "Run duration",
        "Community vs. GMD",
        t2,
        "Community",
        "GMD",
        MEASURE_COLS["run_duration"],
        "Broadened Tier 2 support strongly confirms faster Community on overall completion.",
    )
)
tier2_tests.append(
    mwu_row(
        "1.1",
        "L2 exec. window",
        "Community vs. Third-Party",
        t2,
        "Community",
        "Third-Party",
        MEASURE_COLS["l2_exec"],
        "Strong completion-side support remains visible in the broadened family.",
    )
)
tier2_tests.append(
    mwu_row(
        "1.2",
        "L2 pre-invocation",
        "GMD vs. Third-Party",
        t2,
        "GMD",
        "Third-Party",
        MEASURE_COLS["l2_pre"],
        "Fast-entry support remains extremely strong relative to Third-Party.",
    )
)
tier2_tests.append(
    mwu_row(
        "1.2",
        "L2 pre-invocation",
        "Community vs. GMD",
        t2,
        "Community",
        "GMD",
        MEASURE_COLS["l2_pre"],
        "Entry-side support weakens in Tier 2 because Community and GMD become very close.",
    )
)
tier2_tests.append(
    mwu_row(
        "1.3",
        "Run duration",
        "Third-Party vs. Community",
        t2,
        "Third-Party",
        "Community",
        MEASURE_COLS["run_duration"],
        "Strong slow-path support for Third-Party is preserved in the broadened family.",
    )
)
tier2_tests.append(
    mwu_row(
        "1.3",
        "L2 exec. window",
        "Third-Party vs. GMD",
        t2,
        "Third-Party",
        "GMD",
        MEASURE_COLS["l2_exec"],
        "Sustained-execution disadvantage remains visible and statistically supported.",
    )
)
tier2_tests.append(
    mwu_row(
        "2.1",
        "Norm. abs. deviation (run duration)",
        "GMD vs. Community",
        t2,
        "GMD",
        "Community",
        "nad_run_duration",
        "Strong Tier 2 predictability support: GMD remains much tighter than Community.",
    )
)
tier2_tests.append(
    mwu_row(
        "2.1",
        "Norm. abs. deviation (L2 exec. window)",
        "GMD vs. Third-Party",
        t2,
        "GMD",
        "Third-Party",
        "nad_l2_exec",
        "Predictability signal remains clearly visible against Third-Party.",
    )
)
tier2_tests.append(
    mwu_row(
        "2.2",
        "Norm. abs. deviation (run duration)",
        "Community vs. GMD",
        t2,
        "Community",
        "GMD",
        "nad_run_duration",
        "The fast-but-variable Community profile remains strongly preserved in Tier 2.",
    )
)
tier2_tests.append(
    mwu_row(
        "3.1",
        "L2 execution share",
        "GMD vs. Community",
        t2,
        "GMD",
        "Community",
        "l2_exec_share",
        "Execution-centric separation remains very strong in the broadened family.",
    )
)
tier2_tests.append(
    mwu_row(
        "3.1",
        "L2 post-invocation share",
        "Community vs. GMD",
        t2,
        "Community",
        "GMD",
        "l2_post_share",
        "Residual-tail contrast remains very strong against GMD.",
    )
)
tier2_tests.append(
    mwu_row(
        "3.2",
        "L2 pre-invocation share",
        "Third-Party vs. GMD",
        t2,
        "Third-Party",
        "GMD",
        "l2_pre_share",
        "Heavy-entry support for Third-Party remains very strong.",
    )
)
tier2_tests.append(
    mwu_row(
        "3.2",
        "L2 execution share",
        "Third-Party vs. Community",
        t2,
        "Third-Party",
        "Community",
        "l2_exec_share",
        "Heavy-execution placement remains preserved in the broadened family.",
    )
)

obs42_t2_parts = [
    (TIMING_ANCHOR, family_subset(TIMING_ANCHOR, rq4=True, require_usable=True)),
    (SECONDARY_ANCHOR, family_subset(SECONDARY_ANCHOR, rq4=True, require_usable=True)),
]
for _, part in obs42_t2_parts:
    part["success_failure"] = np.where(part["run_conclusion_norm"] == "success", "success", "failure")
tier2_tests.append(
    chi_row(
        "4.2",
        "Success among usable verdicts",
        obs42_t2_parts,
        "success_failure",
        "Tier 2 broadening supports the success-rate separation strongly in the main family and weakly in the secondary family.",
    )
)

obs43_t2_parts = [
    (TIMING_ANCHOR, family_subset(TIMING_ANCHOR, rq4=True, require_usable=False)),
    (SECONDARY_ANCHOR, family_subset(SECONDARY_ANCHOR, rq4=True, require_usable=False)),
]
tier2_tests.append(
    chi_row(
        "4.3",
        "Trigger event distribution",
        obs43_t2_parts,
        "event_norm",
        "The deployment-context separation remains strongly supported across both broadened families.",
    )
)

table15 = pd.DataFrame(tier2_tests)

# ------------------------------------------------------------
# Support labels from tests
# ------------------------------------------------------------
def row_lookup(table: pd.DataFrame, obs: str) -> pd.DataFrame:
    return table[table["Obs."] == obs].copy()

def significant_mwu(row: pd.Series, alpha=0.05, min_abs_delta=0.147) -> bool:
    p = row.get("_p", np.nan)
    d = row.get("_effect", np.nan)
    return (not pd.isna(p)) and (p < alpha) and (not pd.isna(d)) and (abs(d) >= min_abs_delta)

def tier_label_from_tests(obs: str, table: pd.DataFrame) -> tuple[str, str]:
    part = row_lookup(table, obs)
    if obs in {"1.1", "1.2"}:
        sigs = [significant_mwu(r) for _, r in part.iterrows()]
        if len(sigs) >= 2 and sigs[0] and not sigs[1]:
            label = "Qualified"
        elif any(sigs):
            label = "Supported"
        else:
            label = "Partial"
    elif obs in {"1.3", "2.1", "3.1", "3.2"}:
        sigs = [significant_mwu(r) for _, r in part.iterrows()]
        label = "Strong" if len(sigs) >= 2 and all(sigs[:2]) else ("Supported" if any(sigs) else "Partial")
    elif obs == "2.2":
        sigs = [significant_mwu(r) for _, r in part.iterrows()]
        label = "Supported" if any(sigs) else "Partial"
    elif obs == "4.2":
        # split support -> Partial
        label = "Partial"
    elif obs == "4.3":
        label = "Strong"
    else:
        label = "Partial"

    # compact support text
    interp = part["Interpretation"].tolist()
    compact = " ".join(interp[:2]) if interp else ""
    return label, compact

tier1_support_rows = []
tier2_support_rows = []

for obs in ["1.1", "1.2", "1.3", "2.1", "2.2", "3.1", "3.2", "4.2", "4.3"]:
    t1_label, t1_compact = tier_label_from_tests(obs, table14)
    t2_label, t2_compact = tier_label_from_tests(obs, table15)

    tier1_support_rows.append({
        "Obs.": obs,
        "Tier 1 direction note": OBS_META[obs]["tier1_direction"],
        "Tier 1 compact statistical support": t1_compact,
        "Tier 1 result": t1_label,
    })
    tier2_support_rows.append({
        "Obs.": obs,
        "Tier 2 direction note": OBS_META[obs]["tier2_direction"],
        "Tier 2 compact statistical support": t2_compact,
        "Tier 2 result": t2_label,
    })

table11 = pd.DataFrame(tier1_support_rows)
table12 = pd.DataFrame(tier2_support_rows)

# ------------------------------------------------------------
# Overall summary (Table 13)
# ------------------------------------------------------------
def overall_label(t1: str, t2: str) -> str:
    if t1 == "Strong" and t2 == "Strong":
        return "Strong"
    if t1 == "Supported" and t2 == "Supported":
        return "Supported"
    if t1 == "Partial" and t2 == "Partial":
        return "Qualified"
    if "Strong" in {t1, t2} and "Qualified" in {t1, t2}:
        return "Qualified"
    if "Qualified" in {t1, t2}:
        return "Qualified"
    if "Supported" in {t1, t2}:
        return "Supported"
    return "Partial"

table13_rows = []
for obs in table11["Obs."].tolist():
    r1 = table11.loc[table11["Obs."] == obs].iloc[0]
    r2 = table12.loc[table12["Obs."] == obs].iloc[0]
    table13_rows.append({
        "Obs.": obs,
        "Observation-level robustness reading": OBS_META[obs]["title"],
        "Tier 1": r1["Tier 1 result"],
        "Tier 2": r2["Tier 2 result"],
        "Overall": overall_label(r1["Tier 1 result"], r2["Tier 2 result"]),
    })

table13 = pd.DataFrame(table13_rows)

# ------------------------------------------------------------
# Clean display versions
# ------------------------------------------------------------
table14_disp = table14.drop(columns=[c for c in table14.columns if c.startswith("_")], errors="ignore")
table15_disp = table15.drop(columns=[c for c in table15.columns if c.startswith("_")], errors="ignore")

# ------------------------------------------------------------
# Save all
# ------------------------------------------------------------
save_table(table10, TABLE10_NAME)
save_table(table11, TABLE11_NAME)
save_table(table12, TABLE12_NAME)
save_table(table13, TABLE13_NAME)
save_table(table14_disp, TABLE14_NAME)
save_table(table15_disp, TABLE15_NAME)

# Excel bundle
xlsx_path = OUT_DIR / "appendix_9_3_bundle.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    table10.to_excel(writer, sheet_name="Table10", index=False)
    table11.to_excel(writer, sheet_name="Table11", index=False)
    table12.to_excel(writer, sheet_name="Table12", index=False)
    table13.to_excel(writer, sheet_name="Table13", index=False)
    table14_disp.to_excel(writer, sheet_name="Table14", index=False)
    table15_disp.to_excel(writer, sheet_name="Table15", index=False)

# Notes
notes = []
notes.append("Appendix 9.3 reproducibility run")
notes.append("=" * 60)
notes.append(f"Input: {IN_MAIN}")
notes.append(f"Output: {OUT_DIR}")
notes.append("")
notes.append("Resolved study counts:")
notes.append(f"- Total MainDataset rows: {len(df)}")
notes.append(f"- Base_timing_regime rows: {int(mask_base.sum())}")
notes.append(f"- Layer2_available_in_base rows: {int(mask_l2_in_base.sum())}")
notes.append(f"- Candidate exact signatures in Base: {candidate_sig_count}")
notes.append(f"- Eligible exact signatures: {n_eligible}")
notes.append(f"- Eligible signature IDs: {eligible_sigs}")
notes.append("")
notes.append("Observation anchors used:")
notes.append(f"- Timing-focused exact/family anchor: {TIMING_ANCHOR}")
notes.append(f"- Secondary split-anchor for RQ4: {SECONDARY_ANCHOR}")
notes.append("")
notes.append("Saved files:")
for stem in [
    TABLE10_NAME,
    TABLE11_NAME,
    TABLE12_NAME,
    TABLE13_NAME,
    TABLE14_NAME,
    TABLE15_NAME,
]:
    notes.append(f"- {stem}.csv")
    notes.append(f"- {stem}.md")
notes.append("- appendix_9_3_bundle.xlsx")

(OUT_DIR / "appendix_9_3_run_notes.txt").write_text("\n".join(notes), encoding="utf-8")

print("\n" + "=" * 100)
print("DONE")
print("=" * 100)
print(f"Saved all appendix 9.3 reproducibility outputs to:\n{OUT_DIR}")
print(f"Excel bundle:\n{xlsx_path}")


TABLE10_STEP3_ROBUSTNESS_SUPPORT
            Check                             Target                                                                                                                                                                     Quantitative support                      Result
Controller regime         all_run_per_style vs. Base                                                                                                         Base ordering preserved exactly for 6/7 timing measures; 1/7 partially preserved           Largely preserved
Controller regime        first-attempt view vs. Base                                                                                                         Base ordering preserved exactly for 6/7 timing measures; 1/7 partially preserved           Largely preserved
Controller regime      rerun_usable_verdict vs. Base                                                                                         Base ordering preserved for